In [ ]:
import re
import json
import numpy as np

In [ ]:
subset = "hard-subset"
subset_synonym = "hard-subset"
threshold = 0.95
include_content_similarity_features = True
include_linguistic_features = False

In [ ]:
if not (include_content_similarity_features or include_linguistic_features):
    raise ValueError("At least one of include_content_similarity_features or include_linguistic_features must be True")

In [ ]:
SIMILARITY_FEATURES_FILE = f"[SIMILARITY_FEATURES_SOURCE]"

LINGUISTIC_FEATURES_FILE = f"[LINGUISTIC_FEATURES_SOURCE]" # IGNORE


if include_linguistic_features:
    with open(LINGUISTIC_FEATURES_FILE, "r") as fin:
        linguistic_features_data = json.load(fin)

def extract_paper_fp_from_review_fp(review_filepath):
    # print(review_filepath)
    ## extract the paper contents 
    if '_fragment_' in review_filepath:
        review_filepath = review_filepath.split('_fragment_')[0]
        
    pattern = r".*cleandata/(.*)/(train|test|dev)/.*(level[1-4]|reviews)/(.*)_([1-9]*).txt"
    match = re.search(pattern, review_filepath)
    
    if match is not None:
        conference = match.group(1)
        split = match.group(2)
        level = match.group(3)
        paper_number = match.group(4)
        reviewer_number = match.group(5)

        # return conference, split, level, paper_number, reviewer_number
        generating_model = "OLD PARSER FUNCTION: GENRATING MODEL NOT PARSED"
        prompt = f"{level}@NAV" if level != "reviews" else "DIVINE BENEVOLENCE"

    else:
        pattern = rf".*(?:{subset}|{subset_synonym})/(.*)/(train|test|dev)/(.*)/(level[1-4]|reviews)/(.*).txt"
        match = re.search(pattern, review_filepath)

        conference = match.group(1)
        split = match.group(2)
        paper_number = match.group(3)
        level = match.group(4)

        if '/' in match.group(5):
            generating_model = match.group(5).split('/')[0]
            fileid = match.group(5).split('/')[1]
        else:
            generating_model = "human_review"
            fileid = match.group(5)

        if ":::" in fileid:
            reviewer_number = fileid.split(":::")[-1]
            prompt = fileid.split(":::")[0]
        else:
            reviewer_number = fileid
            if level != "reviews":
                prompt = f"{level}@NAV"
            else:
                prompt = "DIVINE BENEVOLENCE"

    return conference, split, level, paper_number, reviewer_number, generating_model, prompt

if include_linguistic_features and not include_content_similarity_features:
    with open(LINGUISTIC_FEATURES_FILE, "r") as fin:
        data = json.load(fin)
else:
    with open(SIMILARITY_FEATURES_FILE, "r") as fin:
        data = json.load(fin)

scores = {}
counts = {
    'dev': set(),
    'test': set(),
    'train': set()
}

for key, val in data.items():
    conference, split, level, paper_number, reviewer_number, generating_model, prompt = extract_paper_fp_from_review_fp(key)

    counts[split].add((paper_number, conference))

    if level == 'reviews':
        level = 'human'

    if level not in scores.keys():
        scores[level] = []

    if generating_model == "OLD PARSER FUNCTION: GENRATING MODEL NOT PARSED":
        if "gpt" in key:
            generating_model = "gpt_4o_latest"
        elif "llama" in key:
            generating_model = "meta-llama-Llama-3.3-70B-Instruct"
        else:
            generating_model = "human_review"

    # print(conference, split, level, paper_number, reviewer_number, generating_model, prompt)

    record = {
        "id": key,
        "model": generating_model,
        "category": level,
        # "text": open(key.replace("/home/rounak/Downloads/hard-subset", f"/home/rounak/ai-involvement-in-peer-reviews/Data_Preprocessing/cleandata/{subset}"), "r").read().strip(),
        "set": split,
        "paper_number": paper_number,
        "conference": conference, 
        "generating_model": generating_model,
        "prompt": prompt # this is either the prompt id in the form levelk@{id} or if the review is human written it contains a garbage string
    }


    features_dict = dict()
    for feature_name, feature_val in val.items():
        if include_content_similarity_features:
            features_dict[feature_name] = feature_val[str(threshold)] if isinstance(feature_val, dict) else feature_val
        else:
            features_dict[feature_name] = feature_val

    if include_content_similarity_features and not include_linguistic_features:
        features_list = sorted([val for key, val in features_dict.items()])


        record["Feature_Scores"] = {
            f"feat_{i}": features_list[i] for i in range(len(features_list))
        }

    # record["Feature_Scores"]["feature_variance"] = np.var(features_list)
    # record["Feature_Scores"]["feature_mean"] = np.mean(features_list)
    # record["Feature_Scores"]["feature_min"] = np.min(features_list)
    # record["Feature_Scores"]["feature_max"] = np.max(features_list)
    # print(record["Feature_Scores"])

    # i know this is ugly, but i am lazy (EDIT: it is now uglier)
    if include_linguistic_features and include_content_similarity_features:
        record["Feature_Scores"].update(linguistic_features_data[key])

    if include_linguistic_features and not include_content_similarity_features:
        record["Feature_Scores"] = linguistic_features_data[key]

    scores[level].append(record)

# make sure no overlap between train/dev/test
train_dev_overlap = counts['train'].intersection(counts['dev'])
train_test_overlap = counts['train'].intersection(counts['test'])
dev_test_overlap = counts['dev'].intersection(counts['test'])

if not train_dev_overlap and not train_test_overlap and not dev_test_overlap:
    print("No overlap found between train/dev/test sets.")
else:
    print("Overlap found between splits!")
    if train_dev_overlap:
        print(f"\nTrain-Dev overlap ({len(train_dev_overlap)}):")
        for pair in sorted(train_dev_overlap):
            print("  ", pair)
    if train_test_overlap:
        print(f"\nTrain-Test overlap ({len(train_test_overlap)}):")
        for pair in sorted(train_test_overlap):
            print("  ", pair)
    if dev_test_overlap:
        print(f"\nDev-Test overlap ({len(dev_test_overlap)}):")
        for pair in sorted(dev_test_overlap):
            print("  ", pair)


In [ ]:
for category in scores:
    print(f"Category: {category}, Number of items: {len(scores[category])}")

In [ ]:
import pandas as pd

CATEGORY_ORDER = ['level1', 'level2', 'level3', 'level4', 'human']

def prepare_dataframe(scores):
    """
    Convert scores dictionary to pandas DataFrame
    """
    rows = []
    for category, items in scores.items():
        for item in items:
            row = {
                'category': category, 
                'id': item['id'], 
                'set': item['set'], 
                'model': item['model'], 
                'conference': item['conference'],
                'generating_model': item['generating_model'],
                'prompt': item['prompt']
            }
            row.update(item['Feature_Scores'])
            rows.append(row)
    df = pd.DataFrame(rows)
    df['category'] = pd.Categorical(df['category'], categories=CATEGORY_ORDER, ordered=True)
    return df

df = prepare_dataframe(scores)

### Filter examples (by generating model, prompt etc)

In [ ]:
import copy

def filter_examples(df, select_dict, unselect_dict):
    """
    Filter out examples whose 'id' contains the remove_keyword
    """
    filtered_df = copy.deepcopy(df)
    for col, keep_kw in select_dict.items():
        pattern = "|".join(map(re.escape, keep_kw))
        filtered_df = filtered_df[filtered_df[col].str.contains(pattern)]
    for col, remove_kw in unselect_dict.items():
        pattern = "|".join(map(re.escape, remove_kw))
        filtered_df = filtered_df[~filtered_df[col].str.contains(pattern)]
    return filtered_df

### Train the classifier

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import joblib

conferences = df["conference"].unique()

cols_to_drop = ['category', 'id', 'set', 'model', 'conference', 'generating_model', 'prompt'] + [
    col for col in df.columns if ('count' in col.lower())
]

# --- Encode labels once globally ---
le = LabelEncoder()
le.fit(df["category"])

# --- Prepare global test set ---
test_df = df[df["set"] == "test"]

# test_df = filter_examples(
#     test_df,
#     to_be_dict,
# 	not_to_be_dict
# )

X_test = test_df.drop(columns=cols_to_drop)
y_test = le.transform(test_df["category"])

print(f"\nTraining model...")

# --- Split data ---
train_df = df[df["set"] == "train"]

# train_df = filter_examples(
# 	train_df,
# 	to_be_dict,
# 	not_to_be_dict
# )

X_train = train_df.drop(columns=cols_to_drop)
print(f"{len(X_train.columns)} features used for training: {X_train.columns.tolist()}")
y_train = le.transform(train_df["category"])

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)

print("Original:", pd.Series(y_train).value_counts().to_dict())
print("Balanced:", pd.Series(y_train_bal).value_counts().to_dict())

clf = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric='mlogloss',
    tree_method='hist',
    device="cuda:6"
)
clf.fit(X_train_bal, y_train_bal)

# --- Evaluate on global test set ---
test_preds = clf.predict(X_test_scaled)

if '_fragment_' in test_df['id'].iloc[0]:
    ordered_ids = test_df['id'].tolist()
    rev2info = dict()

    for id, true_label, pred_label in zip(ordered_ids, y_test, test_preds):
        rev = id.split('_fragment_')[0] if '_fragment_' in id else id
        if rev not in rev2info:
            rev2info[rev] = dict()
            rev2info[rev]["predictions"] = []
        rev2info[rev]["true_label"] = true_label
        rev2info[rev]["predictions"].append(pred_label)

    # do a majority vote over the predictions for each review
    for rev in rev2info.keys():
        rev2info[rev]["final_prediction"] = np.bincount(rev2info[rev]["predictions"]).argmax()

    y_test_temp = []
    test_preds_temp = []

    for rev in rev2info.keys():
        y_test_temp.append(rev2info[rev]["true_label"])
        test_preds_temp.append(rev2info[rev]["final_prediction"])

    y_test = np.array(y_test_temp)
    test_preds = np.array(test_preds_temp)

test_acc = accuracy_score(y_test, test_preds)
test_f1 = f1_score(y_test, test_preds, average='weighted')

# --- Confusion Matrix for test set ---
cm_labels_order = ['level1', 'level2', 'level3', 'level4', 'human']
cm = confusion_matrix(y_test, test_preds, labels=le.transform(cm_labels_order))
# normalize the confusion matrix row-wise, so that cm_normalized[i][j] = percentage of true class i examples predicted as class j
cm_normalized = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
cm_normalized = cm_normalized.round(2)


# Plot confusion matrix
fig, ax = plt.subplots(figsize=(6,6))
im = ax.imshow(cm_normalized, interpolation='nearest', cmap='Blues', vmin=0, vmax=100)
ax.set_title('Confusion matrix (test predictions)')
ax.set_xticks(np.arange(len(cm_labels_order)))
ax.set_yticks(np.arange(len(cm_labels_order)))
ax.grid(False)
ax.set_xticklabels(cm_labels_order, rotation=45, ha='right')
ax.set_yticklabels(cm_labels_order)
for i in range(cm_normalized.shape[0]):
    for j in range(cm_normalized.shape[1]):
        ax.text(j, i, cm_normalized[i, j], ha='center', va='center', color='white' if cm_normalized[i,j] > cm_normalized.max()/2 else 'black')
# fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


print(f"   Saved model, scaler, encoder, and confusion matrix for threshold = {threshold}")
print(f"   Test Accuracy: {test_acc:.3f}, F1: {test_f1:.3f}")


### Leave-model-out test

In [ ]:
import json
import pandas as pd

OOD_LINGUISTIC_FEATURES_FILE = "/home/rounak/ai-involvement-in-peer-reviews/linguistic_features/extracted_features/hard-subset/linguistic_features_combined-hard-split-main-paper.json"

with open(OOD_LINGUISTIC_FEATURES_FILE, "r") as fin:
    ood_review2features = json.load(fin)

rows = []
for review in ood_review2features.keys():
    category = 'human'

    for i in range(1,5):
        if f'level{i}' in review:
            category = f'level{i}'
            break
    
    entry = {
        'id': review,
        'category': category
    }

    entry.update(ood_review2features[review])
    rows.append(entry)

CATEGORY_ORDER = ["level1", "level2", "level3", "level4", "human"]
ood_df = pd.DataFrame(rows)
ood_df['category'] = pd.Categorical(ood_df['category'], categories=CATEGORY_ORDER, ordered=True)

ood_df = ood_df[ood_df["id"].str.contains('/test/')]
ood_df = ood_df[~ood_df["id"].str.contains('temp')]

cols_to_drop = [
    col for col in ood_df.columns if 'count' in col.lower()
]

# ood_df = ood_df.drop(columns=cols_to_drop)
test_df = ood_df

# select model
test_df = test_df[test_df["id"].str.contains("/gpt-5/")]

# # select base prompt
# test_df = test_df[test_df["id"].str.contains("NAV")]
# test_df = test_df[~test_df["id"].str.contains("NAV_2")]

# select other prompts
test_df = test_df[~test_df["id"].str.contains("NAV:::")]


print(len(test_df))
# print(len(X_test.columns))

In [ ]:
X_test = test_df.drop(columns=cols_to_drop)
y_test = le.transform(test_df["category"])

X_test = X_test.drop(columns=['id', 'category'])
print(len(X_test.columns))
X_test_scaled = scaler.transform(X_test)

test_preds = clf.predict(X_test_scaled)

test_acc = accuracy_score(y_test, test_preds)
test_f1 = f1_score(y_test, test_preds, average='weighted')

cm_labels_order = ['level1', 'level2', 'level3', 'level4', 'human']
cm = confusion_matrix(y_test, test_preds, labels=le.transform(cm_labels_order))
# normalize the confusion matrix row-wise, so that cm_normalized[i][j] = percentage of true class i examples predicted as class j
cm_normalized = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
cm_normalized = cm_normalized.round(2)

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(6,6))
im = ax.imshow(cm_normalized, interpolation='nearest', cmap='Blues', vmin=0, vmax=100)
ax.set_title('Confusion matrix (test predictions)')
ax.set_xticks(np.arange(len(cm_labels_order)))
ax.set_yticks(np.arange(len(cm_labels_order)))
ax.grid(False)
ax.set_xticklabels(cm_labels_order, rotation=45, ha='right')
ax.set_yticklabels(cm_labels_order)
for i in range(cm_normalized.shape[0]):
    for j in range(cm_normalized.shape[1]):
        ax.text(j, i, cm_normalized[i, j], ha='center', va='center', color='white' if cm_normalized[i,j] > cm_normalized.max()/2 else 'black')
plt.tight_layout()
plt.show()

print(f"   Saved model, scaler, encoder, and confusion matrix for threshold = {threshold}")
print(f"   Test Accuracy: {test_acc:.3f}, F1: {test_f1:.3f}")
